[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NOAA-EPIC/EAGLE/blob/main/notebooks/global_mini.ipynb)

# Welcome to the EAGLE pipeline!

Before we start, let's go over a few Google Colab tips!

Q) Where are files located?!

A) You should see a navigation bar on the left of your screen. The 6th option looks like a folder. Click on that and you will see all files in your workspace. If you have not run anything yet, you should only see a "sample data" folder (this automatically populates in any colab notebook). Throughout this notebook you can go into this area and watch your files populate, look at plots, and edit yamls if you wish to update any configurations on your own. Note: sometimes clicking through folders can be a little laggy.

Q) How do I connect to compute?!

A) You will need to connect to a runtime. Towards the top right of your screen you will see the words RAM and disk. There is a drop down button next to that. Click there, and then click "change runtime type". Make sure "Python 3" is selected under runtime type, and if available, select a T4 GPU as your hardware accelerator. If not available, you can run this notebook with a CPU but it will be very slow. If you happen to have credits for an A-100, use that! All estimated run times in this notebook are from using a T4 GPU.

Tip! Google provides free colab credits to use an H-100 for higher education. If that applies to you, go check out [this blog for details](https://blog.google/products-and-platforms/products/education/colab-higher-education/).

Now that you are connected to compute and know where to find your files, let's do some machine learning!

This notebook will guide you through an entire ML pipeline to train and validate a small global model that we call "global-mini". See [EAGLE documentation](https://epic-eagle.readthedocs.io/en/latest/eagle_models/global_mini.html) for more information on global-mini. The various steps of the ML pipeline include:
1) Data preprocessing using `ufs2arco` to create training and validation datasets
2) Model training using `anemoi-core` modules to train a graph-based model
3) Creating a forecast with `anemoi-inference` to run inference from a model checkpoint
4) Verifying your forecast (or multiple!) with `wxvx` to verify against observations

More information about the various modules and instructions will be provided within each individual step. You will also find additional instructions if you wish to change configurations yourself at all!

### Step 1: Environment Setup
Runtime: ~15 minutes

In this step we set up a conda package manager inside Google Colab, clone the EAGLE repository, and create the software environments that the rest of the pipeline depends on.

##### 1a. Install conda in Colab

Colab ships with pip but not conda. The `condacolab` package installs Miniconda and restarts the runtime so that `conda` is available on the path.

> **Note:** You may see a message that says "Your session crashed. Automatically restarting." The session restarts with the fresh conda install. This is expected and not an error. Continue once the restart finishes.

In [ ]:
!pip install -q condacolab
import condacolab

condacolab.install()

##### 1b. Clone the EAGLE repository

We clone the [EAGLE GitHub repository](https://github.com/NOAA-EPIC/EAGLE), which contains the environment files (`envs/`), configuration files (`config/`), and the `eagle` Python modules used throughout this notebook. After this cell runs, you should see the `EAGLE` folder and `sample_data` in the left-hand file browser.


In [ ]:
!git clone https://github.com/NOAA-EPIC/EAGLE.git

In [ ]:
%cd /content/EAGLE/

##### 1c. Create the conda environments

EAGLE uses **four separate conda environments** rather than one large environment. Each environment has a single purpose:

| Environment | Purpose | Environment file |
|---|---|---|
| `base` | Config generation with `uwtools` (`uw config compose` and `uw config realize`) | `envs/base.yaml` |
| `data` | Building training datasets and initial conditions with `ufs2arco` | `envs/data.yaml` |
| `anemoi` | Model training and inference with `anemoi-core` and `anemoi-inference` | `envs/anemoi.yaml` |
| `wxvx` | Forecast verification against gridded analysis and observations | `envs/wxvx.yaml` |

Keeping these separate makes each stack more modular and helps minimize dependency conflicts. Each environment is run with `conda run -n <env_name>`, so you never need to activate them manually inside the notebook.

##### A note on the Anemoi environment

The `anemoi` environment is the only one that depends on the hardware, because model training and inference greatly benefit from a GPU. The setup cell checks for an NVIDIA GPU with `nvidia-smi`:

- **GPU detected:** The environment installs as specified in `envs/anemoi.yaml`, including GPU-only packages such as `triton` and `flash-attn`.
- **No GPU detected:** The cell adjusts the setup for CPU use:
  - Removes `triton` and `flash-attn` from `envs/anemoi.yaml`, since these packages require CUDA.
  - Reduces the training steps in `config/colab.yaml` (`max_steps` and `t_initial`) so that CPU training finishes in a reasonable time.
  - Sets the inference device to `cpu` in `config/colab.yaml` and `config/colab_reforecast.yaml`, so `anemoi-inference` does not try to use a GPU that is not there.

This matters because installing GPU-only packages on a CPU runtime will fail, and running the full training schedule on a CPU would take far longer than the notebook is designed for. The check lets the same notebook run on either hardware type without manual edits.

> **Tip:** The notebook is designed around a T4 GPU. If you only have a CPU runtime, expect training and inference to take significantly longer.

In [ ]:
!conda env update --quiet --name base --file envs/base.yaml

In [ ]:
!conda env create --quiet --name data --file envs/data.yaml

In [ ]:
import shutil

if shutil.which("nvidia-smi"):
    print("GPU Detected. Continue with installation.")
else:
    print("CPU Detected. Installing CPU friendly software versions.")
    # remove packages that require GPU for training and inference (triton and flash-attn)
    !sed -i '/^[[:space:]]*- triton /d; /^[[:space:]]*- flash-attn /d' envs/anemoi.yaml
    # go ahead and greatly reduce number of model steps since CPU training is much slower
    !sed '/max_steps:/s/25/2/; /t_initial:/s/5/1/' -i config/colab.yaml
    # ensure anemoi-inference knows to use device=CPU and CPU friendly precision
    !sed -i '/^inference:$/a\  anemoi:\n    device: cpu' config/colab.yaml
    !sed -i '/^inference:$/a\  anemoi:\n    device: cpu\n    precision: float32' config/colab_reforecast.yaml

!sed 's/- mkl .*/- mkl 2026.1.0.*/' -i envs/anemoi.yaml
!conda env create --quiet --name anemoi --file envs/anemoi.yaml

In [ ]:
!conda env create --quiet --name wxvx --file envs/wxvx.yaml

#### 1d. Install plotting packages and clean up

We install a few extra packages into `base`. These are used for plotting results later in the notebook. Then we finally clear the conda and pip caches. Four environments and their packages take up a large amount of disk space, and Colab runtimes have limited storage. Clearing the caches frees that space without affecting the installed environments.

In [ ]:
# a few packages we need for plotting results
!pip install xarray==2026.2.* matplotlib "zarr<3" "bottleneck>=1.5.0"

In [ ]:
# clean up conda and pip caches to save space
!conda clean --all --yes && pip cache purge

### Step 2: Compose our main config with uwtools
Runtime: Immediate

In this step we build the main configuration file that drives the rest of the pipeline. `uwtools` merges several YAML files into one composed config, so each piece of the setup lives in its own file and can be changed independently.

The `base` environment runs `uw config compose` with three inputs, in order:

| File | Purpose |
|---|---|
| `config/base.yaml` | Settings shared by every EAGLE configuration |
| `config/global_mini.yaml` | Settings specific to the global mini model and dataset |
| `config/colab.yaml` | Settings specific to running in Google Colab, such as reduced training steps and paths |

Later files override values from earlier files when keys overlap. This means `colab.yaml` has the final say for Colab runs.

The composed result is written to `eagle.yaml`. Later steps (data, training, inference, and verification) all read this file, so you do not need to edit it by hand.

In [ ]:
!conda run -n base uw config compose config/base.yaml config/global_mini.yaml config/colab.yaml >eagle.yaml

### Step 3: Create training and validation datasets with ufs2arco

Runtime: ~7 minutes

`ufs2arco` is a Python package developed by the NOAA Physical Sciences Laboratory (PSL). It makes NOAA forecast, reanalysis, and reforecast datasets easier to use for scientific analysis and machine learning. The name reflects its original purpose: transforming output from the Unified Forecast System (UFS) into Analysis Ready, Cloud Optimized (ARCO; Abernathey et al., 2021) format. The package now also pulls data from non-UFS sources, including GFS/GEFS (from before UFS existed) and ECMWF's ERA5 reanalysis.

To learn more about `ufs2arco`, see the [documentation](https://ufs2arco.readthedocs.io/en/latest/index.html).

In this step we create the following dataset:

- **Source:** GFS initial conditions
- **Time step:** 6-hour
- **Training dates:** 2022-01-01T00 to 2022-01-01T18
- **Validation dates:** 2022-01-02T00 to 2022-01-02T18
- **Resolution:** 1-degree global

We are not creating a test set in this notebook, and train on a very small amount of data as an exercise. The results of this model are not expected to make much sense.

The step has two parts:

1. **Create a static regridding file.** GFS data is provided at 0.25-degree resolution. We build a static file that regrids it to 1 degree.
2. **Prepare the training and validation data.** We use ufs2arco to build the dataset in Zarr format.

While the cells are running, go to `/content/EAGLE/run/global_mini/data` and open `logs/logs.serial.out` for more detail on dataset creation. You can also open `ufs2arco-gfs.yaml` to see all data preprocessing settings.

In [ ]:
!conda run -n data uw execute \
  --config-file eagle.yaml \
  --module eagle/data/grids_and_meshes.py \
  --classname GridsAndMeshes \
  --task provisioned_rundir

In [ ]:
!conda run -n data uw execute \
  --config-file eagle.yaml \
  --module eagle/data/zarr.py \
  --classname Zarr \
  --task run \
  --key-path zarrs.gfs

After the dataset has completed, let's view it!

You will notice that this format looks different than a "typical" gridded netCDF or Zarr file. Take note of the flattened grid, statistics, NaN flag, and attributes. These important details make the dataset ready to be used within an ML model.

In [ ]:
import xarray as xr

ufs2arco_ds = xr.open_zarr("/content/EAGLE/run/global_mini/data/gfs.zarr")
ufs2arco_ds

### Step 4: Train a model with anemoi-core modules

Runtime: about 7 minutes ( about 20 minutes on CPU)

We train a graph-based model with the `anemoi-core` modules from the European Centre for Medium-Range Weather Forecasts (ECMWF). The modules include the following:
- **`anemoi-graphs`:** Builds the graph structure that connects the input grid to the model's internal representation. [Documentation](https://anemoi.readthedocs.io/projects/graphs/en/latest/)
- **`anemoi-models`:** Defines the neural network architecture. [Documentation](https://anemoi.readthedocs.io/projects/models/en/latest/)
- **`anemoi-training`:** Runs the training loop and writes checkpoints. [Documentation](https://anemoi.readthedocs.io/projects/training/en/latest/)

Training is executed using `anemoi-training`.

While training is running, go to the `/content/EAGLE/run/global_mini/training/outputs` folder. You will see folders containing checkpoints and plots from your run.

In [ ]:
!conda run -n anemoi uw execute \
  --config-file eagle.yaml \
  --module eagle/training/training.py \
  --classname Training \
  --task run

### Step 5: Create a forecast with anemoi-inference

Runtime: ~20 seconds (will take a few minutes if on a CPU)

See [anemoi-inference documentation](https://anemoi.readthedocs.io/projects/inference/en/latest/) for any details.

In this step we run inference with `anemoi-inference`, using `eagle-tools` to drive it. The forecast covers a 12-hour period, from 2022-01-02T00 to 2022-01-02T12, and starts from the model checkpoint we just trained for a very limited amount of steps.

The output is written to `run/global_mini/inference/output`.

In [ ]:
!conda run -n anemoi uw execute \
  --config-file eagle.yaml \
  --module eagle/inference/inference.py \
  --classname Inference \
  --task run

View inference

Feel free to change the variable or forecast hour that you plot! You can see variable options within the opened dataset below. Simply change the `variable_to_plot` or `fhr_index` variables to update.

In [ ]:
import xarray as xr
ds = xr.open_dataset(
    "/content/EAGLE/run/global_mini/inference/output/2022-01-02T00.12h.nc",
     decode_timedelta=True
)
ds

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

variable_to_plot = "t2m"
fhr_index = 2

var = ds[variable_to_plot].isel(time=fhr_index).values
lat = ds['latitude'].values
lon = ds['longitude'].values

plt.figure(figsize=(10, 6))
plt.scatter(lon, lat, c=var, s=10, marker='s', cmap='coolwarm')
plt.colorbar(label=variable_to_plot)
plt.title(f'{variable_to_plot} at {ds["time"][fhr_index].values}')
plt.show()

### Step 6: Run a reforecast with a pretrained global-mini model

In this step we replace the model we trained above with a pretrained global-mini checkpoint. We then run a reforecast from a date in the testing period (2026) and verify the output against observations.

Pretrained checkpoints depend on the specific `anemoi` package versions used to train them. The EAGLE environment files install the latest versions, which may not match an older checkpoint. The next cell pins the `anemoi` packages in the `anemoi` environment to the versions used to train this checkpoint in October 2026.

Note that the model we trained earlier in this notebook only ran for a short number of steps (25 on a GPU and 2 on a CPU) so that you could work through the pipeline quickly. This pretrained model was trained for 60,000 steps, so its forecasts should be somewhat skillful.

In [ ]:
!conda run -n anemoi pip install \
  anemoi-datasets==0.5.45 \
  anemoi-transform==0.5.0 \
  anemoi-utils==0.5.13

Download the pretrained global-mini checkpoint from Azure Blob Storage.

In [ ]:
!wget -O global-mini.ckpt https://eaglecheckpoints.blob.core.windows.net/eagle-checkpoints/global-mini/global-mini.sanitised.ckpt

Set date to an initialization during the testing period (2026).

You may change this to any date in 2026 that you wish. The rest of the cells in the notebook will adapt to the new date automatically.

In [ ]:
import os

# NOTE: chosen initialization time must be 00Z, 06Z, 12Z, or 18Z.
os.environ["INIT_CHOICE"] = "2026-09-01T00"

Compose the main config for creating a reforecast. This writes `eagle-reforecast-composed.yaml` using `config/colab_reforecast.yaml`, so run it after setting the initialization date above.

In [ ]:
!conda run -n base uw config compose --realize config/base.yaml config/global_mini.yaml config/colab_reforecast.yaml >eagle-reforecast-composed.yaml

Realize the config

Next, we resolve the composed config into a final, fully specified config file.

The composed file (`eagle-reforecast-composed.yaml`) still contains placeholders, such as Jinja2 templates (`{{ ... }}`) and references to other keys, that are only filled in when the config is realized. Values like the forecast initialization time, which we set with `os.environ["INIT_CHOICE"]` above, and the paths built from it are resolved here. This is why the environment variable must be set before this cell runs.

`uw config realize` reads the composed file, evaluates every placeholder and reference, and writes `eagle-reforecast.yaml`. This is the file the rest of the reforecast steps read: loading initial conditions, running inference, and verification. Each later step uses concrete values, not templates.

In [ ]:
!conda run -n base uw config realize \
    --input-file eagle-reforecast-composed.yaml \
    --output-file eagle-reforecast.yaml \
    --output-format yaml

Swap in the correct checkpoint path

The realized config (`eagle-reforecast.yaml`) points to a checkpoint location that is generated from the composed settings. This cell replaces that location with the pretrained checkpoint downloaded earlier, `global-mini.ckpt`, so inference uses the pretrained global-mini model instead of the model trained earlier in this notebook.

In [ ]:
%%bash
cat <<EOF >update.yaml
inference:
  anemoi:
    checkpoint_path: /content/EAGLE/global-mini.ckpt
  checkpoint_dir: !remove
EOF
conda run -n base uw config realize -i eagle-reforecast.yaml -o eagle-reforecast.yaml -u update.yaml

### Load initial conditions

This step prepares the initial conditions for the reforecast. We use the same tools as the training data step, `ufs2arco` run through the `data` environment with `uw execute`, so the commands look familiar. The difference is what we ask for. Training data covered a range of dates and was used to fit the model. Here we only load the initial conditions for the forecast start time that we set with `INIT_CHOICE`.

The `anemoi_inference_with_forcings` target in `ufs2arco` is used for this dataset. Standard initial conditions only include the weather variables that the model predicts. A model also needs static variables, such as the land-sea mask (`lsm`), and forcing variables, such as time-of-day or day-of-year, to stay available for the entire forecast. This target keeps those variables filled in across the forecast period.

When you open the resulting dataset, you will notice:

- **Initial time step:** Weather variables such as `t2m` contain values at the forecast start time.
- **Later time steps:** Weather variables contain `NaN` values, because the model fills them in during inference.
- **Static and forcing variables:** Variables such as `lsm` and the forcings contain real values at every time step.

The step has same two parts as the prepare training data step earlier in the notebook:

1. **Provision the run directory.** `GridsAndMeshes` with `--task provisioned_rundir` creates the grid and mesh files for the reforecast run directory.
2. **Build the initial condition Zarr.** `Zarr` with `--key-path zarrs.gfs` writes the dataset to `nrt_inference/<YYYY/MM/DD/HH>/data/gfs.zarr`.

> **Note about the plot in this section:** We plot the initial conditions for the land-sea mask and `t2m`. The `lsm` row should show values at all three times, while the `t2m` row should show values only for the initial conditions. In global-mini we trained the model to require 2 timesteps for initial conditions instead of 1.

In [ ]:
!conda run -n data uw execute \
  --config-file eagle-reforecast.yaml \
  --module eagle/data/grids_and_meshes.py \
  --classname GridsAndMeshes \
  --task provisioned_rundir

In [ ]:
!conda run -n data uw execute \
  --config-file eagle-reforecast.yaml \
  --module eagle/data/zarr.py \
  --classname Zarr \
  --task run \
  --key-path zarrs.gfs

In [ ]:
import datetime
import os

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

forecast_init_time = datetime.datetime.fromisoformat(os.environ["INIT_CHOICE"])
path = f"/content/EAGLE/run/global_mini/nrt_inference/{forecast_init_time:%Y/%m/%d/%H}/data/gfs.zarr"

ds = xr.open_dataset(
    path,
    decode_timedelta=True,
    engine="zarr",
)

# Build variable name -> index from the dataset attributes
variables = list(ds.attrs["variables"])
variable_index_by_name = {name: idx for idx, name in enumerate(variables)}

# Variables to plot, by name
variable_names_to_plot = ["lsm", "t2m"]
variable_indices = [variable_index_by_name[name] for name in variable_names_to_plot]
variable_names = {variable_index_by_name[name]: name for name in variable_names_to_plot}

# Settings
time_indices = [0, 1, 2]

lat = ds["latitudes"].values
lon = ds["longitudes"].values

fig, axes = plt.subplots(2, 3, figsize=(21, 12), sharex=True, sharey=True)

for row, variable_index in enumerate(variable_indices):
    # Get variable name from dictionary
    var_name = variable_names[variable_index]

    # Shared color scale within each row
    data = ds["data"].isel(time=time_indices, variable=variable_index, ensemble=0)

    vmin = data.min().item()
    vmax = data.max().item()

    for col, t in enumerate(time_indices):
        ax = axes[row, col]

        var = ds["data"].isel(time=t, variable=variable_index, ensemble=0).values

        sc = ax.scatter(
            lon, lat, c=var, s=10, marker="s", cmap="coolwarm", vmin=vmin, vmax=vmax
        )

        ax.set_title(f"{var_name} | Timestep {t}: {ds['dates'].isel(time=t).values}")
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

    # Separate colorbar for each variable
    fig.colorbar(sc, ax=axes[row, :], label=var_name, shrink=0.8, pad=0.02)

### Run inference

Similar to Step 5 above, this step runs `anemoi-inference` on the reforecast, using the pretrained global-mini checkpoint and the initial conditions we just built.

We then animate the results into a GIF to view the 48-hour forecast.

In [ ]:
!conda run -n anemoi uw execute \
  --config-file eagle-reforecast.yaml \
  --module eagle/inference/inference.py \
  --classname Inference \
  --task run

In [ ]:
import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from IPython.display import Image, display

path = (
    f"/content/EAGLE/run/global_mini/nrt_inference/"
    f"{forecast_init_time:%Y/%m/%d/%H}/inference/output/"
    f"{forecast_init_time:%Y-%m-%dT%H}.48h.nc"
)

ds = xr.open_dataset(
    path,
    decode_timedelta=True,
)

variable_to_plot = "t2m"

lat = ds["latitude"].values
lon = ds["longitude"].values
data = ds[variable_to_plot]

vmin = data.min().item()
vmax = data.max().item()

fig, ax = plt.subplots(figsize=(10, 6))

sc = ax.scatter(
    lon,
    lat,
    c=data.isel(time=0).values.ravel(),
    s=10,
    marker="s",
    cmap="coolwarm",
    vmin=vmin,
    vmax=vmax,
)

fig.colorbar(sc, ax=ax, label=variable_to_plot)


def update(t):
    var = data.isel(time=t).values.ravel()
    sc.set_array(var)
    ax.set_title(f"{variable_to_plot} at {ds['time'][t].values}")
    return (sc,)


ani = animation.FuncAnimation(
    fig,
    update,
    frames=ds.sizes["time"],
    interval=500,  # milliseconds between frames
    blit=False,
)

ani.save(f"{variable_to_plot}_animation.gif", writer="pillow", fps=2)
plt.close(fig)

# Display GIF in Jupyter / Colab
display(Image(filename=f"{variable_to_plot}_animation.gif"))

### Verify the forecast
Runtime: ~6 minutes

This step verifies the reforecast against observations using `wxvx`, a verification package that compares forecast output to either gridded analysis or observations.

The cell runs `wxvx` through the `wxvx` environment with `uw execute`, reading the same `eagle-reforecast.yaml` file used in the previous steps. The `vx.grid2obs.global` key tells `wxvx` to compare the gridded forecast against observations on the global domain.

The output includes verification statistics and plots, such as ME and RMSE by lead time and variable. Results are written to the `vx` output folder inside the reforecast run directory.

In [ ]:
# colab does not automatically come with "time", which is required by wxvx.
!apt install -y time

In [ ]:
!conda run -n wxvx uw execute \
  --config-file eagle-reforecast.yaml \
  --module eagle/wxvx/wxvx.py \
  --classname WXVX \
  --task run \
  --key-path vx.grid2obs.global


### Display verification plots

This cell shows the verification plots produced by `wxvx` for the reforecast. It builds the plot folder from `INIT_CHOICE`, so the date and hour in the path update automatically when you change the initialization time.

Each plot name follows the pattern `<variable>-<level>-<metric>-plot.png`. For example:

- `2t-heightAboveGround-0002-RMSE-plot.png` is 2-meter temperature, root mean square error.
- `t-isobaricInhPa-0850-ME-plot.png` is temperature at 850 hPa, mean error.

Provided metrics for each variable are `RMSE` and `ME`.

To choose plots, edit `plot_keywords`:

- `["2t"]` shows all 2-meter temperature plots.
- `["RMSE"]` shows the RMSE plot for every variable.
- `["t-isobaricInhPa-0850", "ME"]` shows only the 850 hPa temperature bias plot.
- `[]` shows every plot in the folder.

In [ ]:
import glob
import os

from IPython.display import Image, display

# Folder built from INIT_CHOICE, so no paths need editing
plot_dir = (
    f"/content/EAGLE/run/global_mini/nrt_inference/"
    f"{forecast_init_time:%Y/%m/%d/%H}/vx/grid2obs/global/run/plots/"
    f"{forecast_init_time:%Y%m%d/%H}"
)

available = sorted(
    os.path.basename(p) for p in glob.glob(os.path.join(plot_dir, "*.png"))
)

print("Available plots:")
for name in available:
    print(f"  {name}")

# Choose plots by keyword. A plot is shown only if its filename contains every keyword.
# Examples: ["2t"], ["RMSE"], ["t-isobaricInhPa-0850", "ME"], [] for all plots
plot_keywords = ["2t", "RMSE"]

selected = [
    name
    for name in available
    if all(keyword.lower() in name.lower() for keyword in plot_keywords)
]

print(f"\nShowing {len(selected)} plot(s):")
for name in selected:
    print(name)
    display(Image(filename=os.path.join(plot_dir, name), width=800))
